In [ ]:
import scipy
import glob
from pathlib import Path
from matplotlib import pyplot as plt
import numpy as np

In [ ]:
# load paths for all EDS sequences
# expects data/processed/<sequence_name>/relative_motions.txt and <sequence_name>.txt
gt_rel_motions_paths = [Path(p) for p in glob.glob('../data/processed/*/relative_motions.txt')]

net_rel_motions_paths = []
for gt_path in gt_rel_motions_paths:
    net_pred = gt_path.parent / (gt_path.parent.name + '.txt')
    net_rel_motions_paths.append(net_pred)

In [ ]:
# dict with sequence name as key and tuple of (gt_rel_motions, net_rel_motions, optimal_scale) as value
seq_data = {}

# calculate scale for minimal RMSE
for gt, net in zip(gt_rel_motions_paths, net_rel_motions_paths):
    
    # t0_us t1_us t1_us px py pz qx qy qz qw
    gt_rel_motions = np.loadtxt(gt, skiprows=1)[:, 2:5]

    # t0_us t1_us px py pz sigma_x sigma_y sigma_z
    net_rel_motions = np.loadtxt(net, skiprows=1)[:, 2:5]

    if gt_rel_motions.shape[0] != net_rel_motions.shape[0]:
        raise ValueError('Number of relative motions in gt and net do not match')

    seq_data[net.parent.name] = (
        gt_rel_motions,
        net_rel_motions,
        np.sum(gt_rel_motions * net_rel_motions) / np.sum(net_rel_motions ** 2),
        )

In [ ]:
# save the data
for seq in net_rel_motions_paths:
    gt_rel_motions, net_rel_motions, optimal_scale = seq_data[seq.parent.name]

    # t0_us t1_us px py pz sigma_x sigma_y sigma_z
    net_original = np.loadtxt(seq, skiprows=1)

    net_rel_motions_scaled = np.concat((net_original[:, :2],
                                       net_original[:, 2:5] * optimal_scale,
                                       net_original[:, 5:]),
                                       axis=1,
                                       )
        
    np.savetxt(seq.parent / (seq.parent.name + '_scaled_rmse.txt'), net_rel_motions_scaled, delimiter=' ', fmt=f"%d %d {' '.join(['%.10f']*6)}", header='t0_us t1_us px py pz sigma_x sigma_y sigma_z', comments='')

In [ ]:
for seq_to_inspect in seq_data.keys():
    gt_rel_motions, net_rel_motions, optimal_scale = seq_data[seq_to_inspect]

    old_rmse = np.sqrt(np.mean((gt_rel_motions - net_rel_motions) ** 2))
    new_rmse = np.sqrt(np.mean((gt_rel_motions - optimal_scale * net_rel_motions) ** 2))

    print(f'Sequence: {seq_to_inspect}, old RMSE: {old_rmse:.4f}, new RMSE: {new_rmse:.4f}')

    plt.figure(figsize=(10, 8))
    plt.title(f'Sequence: {seq_to_inspect}, old RMSE: {old_rmse:.4f}, new RMSE: {new_rmse:.4f}')
    plt.subplot(3,1,1)
    t = np.arange(gt_rel_motions.shape[0])
    plt.plot(t, gt_rel_motions[:, 0])
    plt.plot(t, net_rel_motions[:, 0])
    plt.plot(t, optimal_scale * net_rel_motions[:, 0])
    plt.legend(['gt: ', 'net', f'scaled net {optimal_scale:.2f}'])
    plt.xlabel('Time')
    plt.ylabel('rel x')

    plt.subplot(3,1,2)
    plt.plot(t, gt_rel_motions[:, 1])
    plt.plot(t, net_rel_motions[:, 1])
    plt.plot(t, optimal_scale * net_rel_motions[:, 1])
    plt.legend(['gt: ', 'net', f'scaled net {optimal_scale:.2f}'])
    plt.xlabel('Time')
    plt.ylabel('rel y')

    plt.subplot(3,1,3)
    plt.plot(t, gt_rel_motions[:, 2])
    plt.plot(t, net_rel_motions[:, 2])
    plt.plot(t, optimal_scale * net_rel_motions[:, 2])
    plt.legend(['gt: ', 'net', f'scaled net {optimal_scale:.2f}'])
    plt.xlabel('Time')
    plt.ylabel('rel z')